# Notebook 1: Validación Topológica y Enrutamiento Inteligente

La hipótesis de este notebook es central para el TFM: no todas las series requieren arquitecturas profundas. Usaremos un vector topológico para enrutar datos simples hacia métodos clásicos de Machine Learning, ahorrando cómputo, y datos complejos a modelos de Deep Learning, maximizando precisión. La validación será empírica: primero extraeremos la huella topológica de varios datasets y después comprobaremos si el enrutador dinámico toma la ruta esperada.

## Índice de contenidos

1. [Extracción de la Huella Topológica](#extraccion-topologica)
2. [Motor de Decisiones (Execution Router)](#execution-router)
3. [Test de Tensión: Justificando las decisiones](#test-tension)

    3.1. [Escenario de Baja Complejidad](#esc-baja-comp)
    
    3.2. [Escenario de Alta Complejidad](#esc-alta-comp)

4. [Conclusiones](#conclusiones)

In [1]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
np.set_printoptions(precision=4, suppress=True)

# Resolvemos la raíz del proyecto
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not ((PROJECT_ROOT / "app").exists() and (PROJECT_ROOT / "src").exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not ((PROJECT_ROOT / "app").exists() and (PROJECT_ROOT / "src").exists()):
    raise FileNotFoundError("No se ha podido localizar la raíz del proyecto a partir del directorio actual.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Forzamos un backend SQLite local para que la importación del router no dependa de PostgreSQL.
os.environ.setdefault("DATABASE_URL", f"sqlite:///{(PROJECT_ROOT / 'notebooks' / 'router_validation.db').as_posix()}")

from src.layer2_routing.evaluator import TopologyEvaluator
from src.layer2_routing.router import ExecutionRouter

DATASETS_DIR = PROJECT_ROOT / "notebooks" / "datasets"
if not DATASETS_DIR.exists():
    raise FileNotFoundError(f"No existe el directorio de datasets esperado: {DATASETS_DIR}")

# 1. Carga del dataset de alumbrado público
df_alumbrado = pd.read_csv(DATASETS_DIR / "alumbrado_1.csv", sep=';', decimal=',')
# Fijar la primera columna como índice de fecha
df_alumbrado.set_index('Fecha', inplace=True)
df_alumbrado.index = pd.to_datetime(df_alumbrado.index, dayfirst=True)

# 2. Carga del dataset de Cups
df_cups_raw = pd.read_csv(DATASETS_DIR / "cups.csv")
df_cups_raw["timestamp"] = pd.to_datetime(df_cups_raw["timestamp"], utc=True).dt.tz_localize(None)
cups_cup = df_cups_raw["cups"].mode().iat[0]
df_cups = (
    df_cups_raw.loc[df_cups_raw["cups"] == cups_cup, ["timestamp", "value"]]
    .sort_values("timestamp")
    .set_index("timestamp")
)

# 3. Carga del dataset de ETT
df_ett = pd.read_csv(
    DATASETS_DIR / "electricity_transformer_temperature" / "ETTh1.csv",
    index_col="date",
    parse_dates=True,
)

# 4. Carga del dataset de PeMS
df_pems = pd.read_csv(DATASETS_DIR / "pems_traffic" / "traffic.txt", header=None)
df_pems.index = pd.date_range("2015-01-01", periods=len(df_pems), freq="h")
df_pems.index.name = "date"
df_pems.columns = [f"sensor_{idx}" for idx in range(df_pems.shape[1])]

pd.DataFrame(
    [
        {"Dataset": "Alumbrado", "Filas": df_alumbrado.shape[0], "Columnas": df_alumbrado.shape[1]},
        {"Dataset": "Cups", "Filas": df_cups.shape[0], "Columnas": df_cups.shape[1]},
        {"Dataset": "ETT", "Filas": df_ett.shape[0], "Columnas": df_ett.shape[1]},
        {"Dataset": "PeMS", "Filas": df_pems.shape[0], "Columnas": df_pems.shape[1]},
    ]
)

,Dataset,Filas,Columnas
0,Alumbrado,8746,1
1,Cups,14065,1
2,ETT,17420,7
3,PeMS,17544,862


<a id="extraccion-topologica"></a>
## 1. Extracción de la Huella Topológica

El `TopologyEvaluator` resume cada dataset en un vector ligero de 3 dimensiones. La primera componente es la **Autocorrelación (ACF)**, calculada mediante un barrido dinámico de lags sobre una ventana de 168 horas (un ciclo semanal completo), que aproxima la previsibilidad temporal. La segunda es la **Curtosis** (exceso de Fisher), que actúa como una medida de volatilidad y eventos extremos. La tercera es la **Correlación Cruzada Absoluta Media** (media del triángulo superior de correlaciones absolutas de Pearson entre todos los pares de canales), que cuantifica el acoplamiento lineal en casos multivariantes. Este resumen permite comparar series muy distintas bajo las mismas métricas.

In [2]:
evaluator = TopologyEvaluator()

start_time = time.perf_counter()
profile_cups = evaluator.extract_topology_profile(df_cups)
vector_cups = profile_cups["topology_vector_3D"]
elapsed_cups = time.perf_counter() - start_time

start_time = time.perf_counter()
profile_alumbrado = evaluator.extract_topology_profile(df_alumbrado)
vector_alumbrado = profile_alumbrado["topology_vector_3D"]
elapsed_alumbrado = time.perf_counter() - start_time

start_time = time.perf_counter()
profile_ett = evaluator.extract_topology_profile(df_ett)
vector_ett = profile_ett["topology_vector_3D"]
elapsed_ett = time.perf_counter() - start_time

start_time = time.perf_counter()
profile_pems = evaluator.extract_topology_profile(df_pems)
vector_pems = profile_pems["topology_vector_3D"]
elapsed_pems = time.perf_counter() - start_time

df_topology = pd.DataFrame(
    [
        {"Dataset": "Alumbrado", "ACF (168h)": vector_alumbrado[0], "Kurtosis": vector_alumbrado[1], "Cross_Corr (Mean |ρ|)": vector_alumbrado[2]},
        {"Dataset": "Cups", "ACF (168h)": vector_cups[0], "Kurtosis": vector_cups[1], "Cross_Corr (Mean |ρ|)": vector_cups[2]},
        {"Dataset": "ETT", "ACF (168h)": vector_ett[0], "Kurtosis": vector_ett[1], "Cross_Corr (Mean |ρ|)": vector_ett[2]},
        {"Dataset": "PeMS", "ACF (168h)": vector_pems[0], "Kurtosis": vector_pems[1], "Cross_Corr (Mean |ρ|)": vector_pems[2]},
    ]
)

print("Tiempos de extracción topológica (s):")
print(f"  - Alumbrado: {elapsed_alumbrado:.4f}")
print(f"  - Cups: {elapsed_cups:.4f}")
print(f"  - ETT: {elapsed_ett:.4f}")
print(f"  - PeMS: {elapsed_pems:.4f}")

df_topology.style.format(
    {"ACF (168h)": "{:.4f}", "Kurtosis": "{:.4f}", "Cross_Corr (Mean |ρ|)": "{:.4f}"}
).background_gradient(
    cmap="YlOrRd",
    subset=["ACF (168h)", "Kurtosis", "Cross_Corr (Mean |ρ|)"],
)

Tiempos de extracción topológica (s):
  - Alumbrado: 0.0515
  - Cups: 0.1028
  - ETT: 0.5675
  - PeMS: 73.7868


,Dataset,ACF (168h),Kurtosis,Cross_Corr (Mean |ρ|)
0,Alumbrado,0.9898,-1.8611,0.0000
1,Cups,0.8433,5.5949,0.0000
2,ETT,0.9714,2.5973,0.2221
3,PeMS,0.9168,18.8657,0.5638


**Análisis de los Vectores Topológicos:**
Los vectores extraídos nos revelan la naturaleza de cada señal:

* **Alumbrado (ACF alta, Curtosis negativa → Bloqueo de Onda Cuadrada):** Su alta autocorrelación (ACF > 0.80) combinada con una curtosis negativa (-1.86, distribución platicúrtica) es la firma de una señal similar a un escalón (onda cuadrada de encendido/apagado). Esta combinación activa la regla de bloqueo de onda cuadrada del router, forzando directamente `ML_CLASSIC` sin evaluar más condiciones.
* **Cups (ACF alta, Curtosis extrema → Gatillo de Complejidad):** Aunque es univariante y estacional, su curtosis de 5.59 (muy por encima del umbral 4.0) activa el gatillo de complejidad univariante. Con volumen suficiente (≥ 2500 muestras), el router la escala a Deep Learning.
* **ETT (Multivariante, Correlación absoluta media débil):** Presenta 7 canales con una alta dependencia temporal (ACF 0.97), pero una correlación absoluta media débil (0.22). Esto indica dependencias no lineales entre canales. Al ser < 0.80, se descarta SoftImpute y se activa la ruta DL (SAITS).
* **PeMS (Multivariante masivo, Correlación absoluta media moderada):** Con 862 sensores y una curtosis altísima (18.86), su correlación absoluta media es de 0.56. Esto refleja que aunque hay fuertes interacciones locales, la matriz global no tiene un acoplamiento lineal perfecto. Al ser < 0.80, el router bloquea la ruta clásica y deriva la señal a `SAITS`.

<a id="execution-router"></a>
## 2. Motor de Decisiones (Execution Router)

Una vez calculado el vector topológico, el `ExecutionRouter` aplica una lógica bifurcada por dimensionalidad. En señales univariantes, evalúa secuencialmente: bloqueo de onda cuadrada (kurt < 0 AND ACF > 0.80), gatillo de complejidad (kurt > 4.0 OR ACF < 0.50), y viabilidad por volumen (≥ 2500 muestras). En señales multivariantes, el único discriminante es la correlación cruzada absoluta media: si ≥ 0.80, SoftImpute basta (`ML_CLASSIC`); si < 0.80, se escala a SAITS (≥ 5000) o TimesNet (≥ 2500). El resultado es una ruta `ML_CLASSIC` o `DL_EPHEMERAL`.

In [3]:
from dataclasses import dataclass, field
from typing import Any, Optional

@dataclass
class MockPreferences:
    quick_response_required: bool = False

@dataclass
class MockContext:
    topology_vector_3D: list[float]
    num_channels: int
    valid_samples: int
    raw_data: pd.DataFrame
    preferences: MockPreferences = field(default_factory=MockPreferences)
    metadata: Optional[dict[str, Any]] = None

router = ExecutionRouter(registry=None)

def simulate_route(
    dataset_name: str,
    df: pd.DataFrame,
    topology_vector: list[float],
) -> dict:
    """
    Simula la decisión del router con un contexto mock.
    """
    mock_context = MockContext(
        topology_vector_3D=topology_vector,
        num_channels=df.shape[1],
        valid_samples=int(df.notna().any(axis=1).sum()),
        raw_data=df,
    )
    requires_dl, architecture = router._requires_dl(mock_context)
    route = "Deep Learning" if requires_dl else "Machine Learning Clásico"
    architecture_label = architecture if architecture is not None else "No aplica"
    print(f"{dataset_name:12s} -> Ruta: {route:25s} | Arquitectura: {architecture_label:15s} | Muestras: {mock_context.valid_samples:>6d}")
    return {
        "Dataset": dataset_name,
        "Ruta": route,
        "Arquitectura": architecture_label,
        "Muestras": mock_context.valid_samples,
    }

routing_results = pd.DataFrame(
    [
        simulate_route("Alumbrado", df_alumbrado, vector_alumbrado),
        simulate_route("Cups", df_cups, vector_cups),
        simulate_route("ETT", df_ett, vector_ett),
        simulate_route("PeMS", df_pems, vector_pems),
    ]
)

Alumbrado    -> Ruta: Machine Learning Clásico  | Arquitectura: No aplica       | Muestras:   8746
Cups         -> Ruta: Deep Learning             | Arquitectura: BiLSTM          | Muestras:  14065
ETT          -> Ruta: Deep Learning             | Arquitectura: SAITS           | Muestras:  17420
PeMS         -> Ruta: Deep Learning             | Arquitectura: SAITS           | Muestras:  17544


**Decisiones del ExecutionRouter:**

El enrutador aplica reglas según la dimensionalidad:

1. **Alumbrado $\rightarrow$ ML_CLASSIC (Bloqueo de Onda Cuadrada):** La curtosis negativa ($\text{kurt} < 0$) combinada con ACF alta ($> 0.80$) activa la regla de bloqueo estricto para señales tipo escalón. El router fuerza `ML_CLASSIC` sin evaluar más condiciones, evitando el gasto computacional innecesario de entrenar un BiLSTM sobre una señal que LightGBM puede resolver de manera más eficiente.

2. **Cups $\rightarrow$ DL (Bi-LSTM):** Su curtosis ($> 4.0$) activa el gatillo de complejidad univariante. Con muestras suficientes ($\geq 2500$), el router asigna `LIGHTWEIGHT_DL`: LSTM Autoencoder para detección y BiLSTM para imputación.
3. **ETT $\rightarrow$ DL (SAITS):** La correlación cruzada absoluta media es baja ($< 0.80$), indicando dependencias no lineales entre los 7 canales. Con muestras suficientes ($\geq 5000$), se asigna SAITS (Transformer con self-attention).
4. **PeMS $\rightarrow$ DL (SAITS):** A pesar de su enorme dimensionalidad (862 canales), su correlación cruzada absoluta media es moderada (0.56, por debajo del exigente umbral de 0.80 que requiere SoftImpute). Al contar con 17,544 muestras ($\geq 5000$), el router despliega `SAITS` para capturar las interacciones espaciales locales del tráfico mediante su mecanismo de auto-atención.

<a id="test-tension"></a>
## 3. Test de Tensión y Validación Empírica

En esta sección validaremos las decisiones con una prueba de **Imputación de Huecos**. Inyectaremos bloques contiguos de valores nulos y compararemos el modelo recomendado por la ruta topológica frente a uno forzado inapropiado. Usaremos exclusivamente clases reales del proyecto y sus hiperparámetros por defecto, la Optimización de Hiperparámetros (HPO) bayesiana se tratará en notebooks posteriores.

In [11]:
import time
import numpy as np
from sklearn.metrics import mean_absolute_error

def inject_missing_values(df, missing_ratio=0.15, block_size=6, random_state=42):
    if df is None or df.empty:
        raise ValueError("El DataFrame de entrada no puede estar vacio.")

    df_corrupto = df.copy(deep=True)
    ground_truth = df.copy(deep=True)

    numeric_cols = df_corrupto.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        raise ValueError("No se encontraron columnas numericas para inyectar NaNs.")

    target_col = numeric_cols[0]
    serie = pd.to_numeric(df_corrupto[target_col], errors="coerce")
    valid_idx = np.flatnonzero(np.isfinite(serie.to_numpy(dtype=float)))
    if valid_idx.size == 0:
        raise ValueError("La primera columna numerica no contiene datos validos.")

    rng = np.random.default_rng(random_state)
    n_missing_target = max(1, int(np.floor(valid_idx.size * float(missing_ratio))))
    block_size = max(1, int(block_size))

    mascara_nulos = pd.Series(False, index=df_corrupto.index, dtype=bool, name=f"mask_{target_col}")
    inserted = 0
    attempts = 0
    max_attempts = valid_idx.size * 20

    while inserted < n_missing_target and attempts < max_attempts:
        attempts += 1
        anchor = int(rng.choice(valid_idx))
        start = max(0, min(anchor, len(df_corrupto) - block_size))
        end = min(len(df_corrupto), start + block_size)

        block_index = df_corrupto.index[start:end]
        candidate = block_index[ground_truth.loc[block_index, target_col].notna()]
        if len(candidate) == 0:
            continue

        available = candidate[~mascara_nulos.loc[candidate]]
        if len(available) == 0:
            continue

        room = n_missing_target - inserted
        chosen = available[:room]
        df_corrupto.loc[chosen, target_col] = np.nan
        mascara_nulos.loc[chosen] = True
        inserted += len(chosen)

    if inserted < n_missing_target:
        remaining_idx = [
            idx
            for idx in ground_truth.index
            if pd.notna(ground_truth.at[idx, target_col]) and not bool(mascara_nulos.at[idx])
        ]
        if remaining_idx:
            topup = remaining_idx[: n_missing_target - inserted]
            df_corrupto.loc[topup, target_col] = np.nan
            mascara_nulos.loc[topup] = True

    return df_corrupto, ground_truth, mascara_nulos

<a id="esc-baja-comp"></a>
### 3.1. Escenario de Baja Complejidad (Alumbrado)

En este escenario usamos el dataset de **Alumbrado** en `notebooks/datasets/alumbrado_1.csv`, cuya ruta recomendada por topología es **ML_CLASSIC** gracias a la regla de **Bloqueo de Onda Cuadrada** (curtosis negativa + ACF > 0.80). El router nos ha salvado de ejecutar un BiLSTM innecesario. Compararemos `LightGBM` (campeón esperado) frente a `BiLSTM` como retador profundo para verificar que la decisión fue acertada.

**Ingeniería de Características**: los modelos tabulares no entienden el tiempo de forma inherente. Por ello usamos `TimeSeriesFeatureExtractor` para generar variables de retardo (lags) y variables de calendario. Además, inyectamos la localización espacial (Cantabria, España) para derivar eventos solares y festivos, dando a LightGBM el contexto físico que necesita para el alumbrado público.

In [12]:
from src.layer3_models.imputer import LightGBMImputer, BiLSTMImputer

# --- 1) Particion cronologica 80/20 (Out-of-Time) ---
if len(df_alumbrado) < 2:
    raise ValueError("df_alumbrado debe tener al menos 2 filas para realizar una particion 80/20.")

split_idx = int(len(df_alumbrado) * 0.8)
split_idx = min(max(split_idx, 1), len(df_alumbrado) - 1)

df_train = df_alumbrado.iloc[:split_idx].copy(deep=True)
df_test = df_alumbrado.iloc[split_idx:].copy(deep=True)

alumbrado_col = df_alumbrado.select_dtypes(include=[np.number]).columns[0]

# --- 2) Inyeccion de huecos solo en test ---
df_test_corrupto, gt_test, mask_test = inject_missing_values(
    df_test.copy(deep=True),
    missing_ratio=0.15,
    block_size=6,
    random_state=42,
)

# --- 3) Preparacion para inferencia con contexto temporal ---
df_inference = pd.concat([df_train, df_test_corrupto], axis=0)

location_config = {
    "country": "ES",
    "region": "Cantabria",
    "city": "Santander",
    "timezone": "Europe/Madrid",
}

# --- 4) Contendiente 1: LightGBM Ligero (fit -> transform) ---
print("[Alumbrado] Entrenando LightGBM (Modo Ligero: Solo lag diario)...")
start_time = time.perf_counter()
lightgbm_ligero = LightGBMImputer(
    lags=[24],
)
lightgbm_ligero.fit(df_train[[alumbrado_col]])
df_alumbrado_lgb_ligero_full = lightgbm_ligero.transform(df_inference[[alumbrado_col]])
elapsed_lgb_ligero = time.perf_counter() - start_time
df_alumbrado_lgb_ligero_test = df_alumbrado_lgb_ligero_full.iloc[split_idx:].copy(deep=True)

# --- 4) Contendiente 2: LightGBM Pesado (fit -> transform) ---
print("[Alumbrado] Entrenando LightGBM (Modo Pesado: Contexto espacial + Lags complejos)...")
start_time = time.perf_counter()
lightgbm_pesado = LightGBMImputer(
    lags=[1, 24, 168],
    rolling_windows=[3, 6, 12],
    location_config=location_config,
 )
lightgbm_pesado.fit(df_train[[alumbrado_col]])
df_alumbrado_lgb_pesado_full = lightgbm_pesado.transform(df_inference[[alumbrado_col]])
elapsed_lgb_pesado = time.perf_counter() - start_time
df_alumbrado_lgb_pesado_test = df_alumbrado_lgb_pesado_full.iloc[split_idx:].copy(deep=True)

# --- 4) Contendiente 3: BiLSTM (fit -> transform) ---
print("[Alumbrado] Entrenando Deep Learning (BiLSTM - 100 epocas)...")
start_time = time.perf_counter()
bilstm_imputer = BiLSTMImputer(epochs=100, random_state=42)
bilstm_imputer.fit(df_train[[alumbrado_col]])
df_alumbrado_bilstm_full = bilstm_imputer.transform(df_inference[[alumbrado_col]])
elapsed_bilstm = time.perf_counter() - start_time
df_alumbrado_bilstm_test = df_alumbrado_bilstm_full.iloc[split_idx:].copy(deep=True)

# --- 5) Evaluacion (solo sobre huecos inyectados en test) ---
y_true_alumbrado = gt_test.loc[mask_test, alumbrado_col]
y_pred_lgb_ligero = df_alumbrado_lgb_ligero_test.loc[mask_test, alumbrado_col]
y_pred_lgb_pesado = df_alumbrado_lgb_pesado_test.loc[mask_test, alumbrado_col]
y_pred_bilstm = df_alumbrado_bilstm_test.loc[mask_test, alumbrado_col]

mae_lgb_ligero = mean_absolute_error(y_true_alumbrado, y_pred_lgb_ligero)
mae_lgb_pesado = mean_absolute_error(y_true_alumbrado, y_pred_lgb_pesado)
mae_bilstm = mean_absolute_error(y_true_alumbrado, y_pred_bilstm)

print("\n" + "=" * 68)
print("RESULTADOS OOT (80/20): TEST DE TENSION ALUMBRADO (SIN DATA LEAKAGE)")
print("=" * 68)
print(f"Train: {df_train.shape[0]} filas | Test: {df_test.shape[0]} filas")
print(f"Huecos inyectados en test: {int(mask_test.sum())}")
print("=" * 68)
print(f"1. LightGBM (Ligero) : Tiempo = {elapsed_lgb_ligero:7.3f} s | MAE = {mae_lgb_ligero:.6f}")
print(f"2. BiLSTM (DL)       : Tiempo = {elapsed_bilstm:7.3f} s | MAE = {mae_bilstm:.6f}")
print(f"3. LightGBM (Pesado) : Tiempo = {elapsed_lgb_pesado:7.3f} s | MAE = {mae_lgb_pesado:.6f}")
print("=" * 68)

[Alumbrado] Entrenando LightGBM (Modo Ligero: Solo lag diario)...
[Alumbrado] Entrenando LightGBM (Modo Pesado: Contexto espacial + Lags complejos)...
[Alumbrado] Entrenando Deep Learning (BiLSTM - 100 epocas)...

RESULTADOS OOT (80/20): TEST DE TENSION ALUMBRADO (SIN DATA LEAKAGE)
Train: 6996 filas | Test: 1750 filas
Huecos inyectados en test: 262
1. LightGBM (Ligero) : Tiempo =   8.241 s | MAE = 0.120237
2. BiLSTM (DL)       : Tiempo = 443.680 s | MAE = 0.380157
3. LightGBM (Pesado) : Tiempo =  39.471 s | MAE = 0.112158


<a id="esc-alta-comp"></a>
### 3.2. Escenario de Alta Complejidad (ETT: Electricity Transformer Temperature)

Ahora probamos **ETT** multivariante, cuya correlación cruzada absoluta media es baja (< 0.80). Según la lógica del router, esta debilidad en el acoplamiento lineal entre canales descarta `SoftImpute` y activa la ruta hacia `SAITS`.

Compararemos ambos métodos para observar el trade-off precisión-latencia en una matriz de 7 sensores.

In [18]:
import time
import warnings
import torch
import numpy as np
import pandas as pd
from pypots.imputation import SAITS
from pypots.optim.adam import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from src.layer3_models.imputer import SoftImputeImputer

# Silenciar el FutureWarning de scikit-learn provocado por fancyimpute
warnings.filterwarnings("ignore", category=FutureWarning)

def _to_pypots_chunks(df, n_steps):
    values = df.to_numpy(dtype=float)
    remainder = values.shape[0] % n_steps
    pad_size = (n_steps - remainder) if remainder != 0 else 0
    if pad_size > 0:
        padding = np.full((pad_size, values.shape[1]), np.nan)
        values = np.vstack([values, padding])
    return values.reshape(-1, n_steps, values.shape[1]), pad_size

def _from_pypots_chunks(values_3d, original_length):
    values_2d = values_3d.reshape(-1, values_3d.shape[2])
    return values_2d[:original_length, :]

print(f"[ETT] Preparando Test de Tensión OOT (80/20) sobre matriz de {df_ett.shape[0]} filas x {df_ett.shape[1]} sensores.")

# --- 1. PARTICIÓN CRONOLÓGICA (OOT) ---
split_idx = int(len(df_ett) * 0.8)
df_train = df_ett.iloc[:split_idx].copy()
df_test = df_ett.iloc[split_idx:].copy()

# --- 2. INYECCIÓN DE HUECOS (Solo en Test) ---
df_test_corrupto, gt_test, mask_test = inject_missing_values(
    df_test,
    missing_ratio=0.15,
    block_size=6,
    random_state=42,
)

# Matriz de inferencia completa (Pasado limpio + Futuro corrupto)
df_inference = pd.concat([df_train, df_test_corrupto])


# --- CONTENDIENTE 1: SoftImpute (ML Clásico Transductivo) ---
print("[ETT] Entrenando ML Clásico (SoftImpute)...")
start_time = time.perf_counter()
soft_impute_imputer = SoftImputeImputer(max_iters=100)
# SoftImpute recibe toda la inferencia junta para hacer Matrix Completion
df_inference_soft_impute = soft_impute_imputer.impute(df_inference)
elapsed_soft_impute = time.perf_counter() - start_time

# Extraemos solo la parte del test para evaluar
df_test_soft_impute = df_inference_soft_impute.iloc[split_idx:]


# --- CONTENDIENTE 2: SAITS (Deep Learning Transformer) ---
print("[ETT] Entrenando Deep Learning (SAITS - 100 épocas)...")
n_steps = 24

# Escalado riguroso (Fit solo en train)
scaler = StandardScaler()
array_train_scaled = scaler.fit_transform(df_train)
array_inference_scaled = scaler.transform(df_inference)

df_train_scaled = pd.DataFrame(array_train_scaled, index=df_train.index, columns=df_train.columns)
df_inference_scaled = pd.DataFrame(array_inference_scaled, index=df_inference.index, columns=df_inference.columns)

# Conversión a 3D
X_train_3d, _ = _to_pypots_chunks(df_train_scaled, n_steps=n_steps)
X_inference_3d, pad_size = _to_pypots_chunks(df_inference_scaled, n_steps=n_steps)

start_time = time.perf_counter()
saits_model = SAITS(
    n_steps=n_steps,
    n_features=df_ett.shape[1],
    epochs=100,     
    patience=10,
    n_layers=3,
    d_model=512,
    d_ffn=128,
    n_heads=4,
    d_k=128,
    d_v=64,
    dropout=0.1,
    attn_dropout=0.1,
    optimizer=Adam(lr=0.001),
    device="cuda" if torch.cuda.is_available() else "cpu",
    model_saving_strategy=None,
)

# Fit solo con Train, Impute con Inferencia
saits_model.fit(train_set={"X": X_train_3d})
X_inference_imputed_3d = saits_model.impute({"X": X_inference_3d})
elapsed_saits = time.perf_counter() - start_time

# Reconstruir a 2D y Desescalar
X_inference_imputed_2d_scaled = _from_pypots_chunks(X_inference_imputed_3d, original_length=len(df_inference))
X_inference_imputed_2d = scaler.inverse_transform(X_inference_imputed_2d_scaled)

df_inference_saits = pd.DataFrame(X_inference_imputed_2d, index=df_inference.index, columns=df_inference.columns)

# Extraemos solo la parte del test para evaluar
df_test_saits = df_inference_saits.iloc[split_idx:]


# --- EVALUACIÓN Y REPORTE ---
# Usamos .to_numpy() para manejar las máscaras booleanas N-dimensionales de forma segura
mask_array = mask_test.to_numpy(dtype=bool)
y_true_ett = gt_test.to_numpy()[mask_array]
y_pred_soft_impute = df_test_soft_impute.to_numpy()[mask_array]
y_pred_saits = df_test_saits.to_numpy()[mask_array]

mae_soft_impute = mean_absolute_error(y_true_ett, y_pred_soft_impute)
mae_saits = mean_absolute_error(y_true_ett, y_pred_saits)

print("\n" + "="*70)
print("RESULTADOS OOT (80/20): TEST DE TENSIÓN ETT (Alta Complejidad)")
print("="*70)
print(f"Train: {len(df_train)} filas | Test: {len(df_test)} filas")
print(f"1. ML Clásico (SoftImpute) : Tiempo = {elapsed_soft_impute:7.3f} s | MAE = {mae_soft_impute:.6f}")
print(f"2. Deep Learning (SAITS)   : Tiempo = {elapsed_saits:7.3f} s | MAE = {mae_saits:.6f}")
print("="*70)

if mae_soft_impute > mae_saits:
    ratio = mae_soft_impute / max(mae_saits, 1e-12)
    print(f"CONCLUSIÓN: El Enrutador acertó. SoftImpute colapsa frente a SAITS (Error x{ratio:.2f} mayor).")
else:
    print("CONCLUSIÓN: SoftImpute resistió el embate. (Revisar hiperparámetros si se busca mayor separación).")

[ETT] Preparando Test de Tensión OOT (80/20) sobre matriz de 17420 filas x 7 sensores.
[ETT] Entrenando ML Clásico (SoftImpute)...


2026-05-26 10:59:01 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.


[ETT] Entrenando Deep Learning (SAITS - 100 épocas)...

RESULTADOS OOT (80/20): TEST DE TENSIÓN ETT (Alta Complejidad)
Train: 13936 filas | Test: 3484 filas
1. ML Clásico (SoftImpute) : Tiempo =   0.115 s | MAE = 0.171419
2. Deep Learning (SAITS)   : Tiempo = 941.988 s | MAE = 0.064346
CONCLUSIÓN: El Enrutador acertó. SoftImpute colapsa frente a SAITS (Error x2.66 mayor).


<a id="conclusiones"></a>
## 4. Conclusiones

El test de tensión demuestra la eficacia del `ExecutionRouter` y su lógica de enrutamiento. Podemos extraer dos conclusiones fundamentales sobre el comportamiento de los modelos algorítmicos:

### 4.1. El Bloqueo de Onda Cuadrada protege contra el colapso de DL (Caso Alumbrado)
En señales de baja complejidad y alta estacionalidad (como la onda cuasi-cuadrada del encendido del alumbrado), las redes neuronales profundas sufren un colapso. El router lo detectó preventivamente: la combinación de curtosis negativa (distribución platicúrtica, sin colas pesadas) y ACF > 0.80 (señal altamente periódica) activó la regla de bloqueo de onda cuadrada, forzando `ML_CLASSIC` sin evaluar más condiciones. El test de tensión confirma que esta decisión fue acertada:
* **Árboles vs. Curvas:** La arquitectura BiLSTM no es la óptima para trazar ángulos muy pronunciados (escalones perfectos) debido a la naturaleza de sus funciones de activación suaves, resultando en un MAE tres veces mayor (0.38) y una latencia 50 veces superior (443 s) frente a un método basado en particiones ortogonales como LightGBM.
* **La trampa de la sobre-ingeniería:** El modelo LightGBM "Pesado" (con 29 variables, incluyendo ventanas rodantes) apenas mejoró el MAE del modelo "Ligero" (0.112 vs 0.120), pero multiplicó su coste temporal casi por cinco (39 s vs 8 s). Al aplicar medias móviles a una onda cuadrada perfecta, se difuminan los bordes de la señal, confundiendo a los árboles de decisión. Esto subraya la necesidad de implementar mecanismos de **Selección de Variables por Importancia**, que se desarrollarán en fases posteriores para mantener el equilibrio entre latencia y precisión.

### 4.2. La Correlación Absoluta Media como discriminante multivariante (Caso ETT)
En el extremo opuesto, cuando nos enfrentamos a matrices multivariantes con dependencias no lineales, nuestros métodos clásicos colapsan. El router lo detectó correctamente: la correlación cruzada absoluta media del ETT era baja (< 0.80), indicando que las relaciones entre los 7 canales no son linealmente explotables por `SoftImpute`. Esto activó la ruta hacia `SAITS`.
* Modelos como `SoftImpute` asumen estructura de bajo rango con acoplamiento lineal fuerte. Al aplicarlo al dataset ETT (cuya correlación cruzada absoluta media era < 0.80), el modelo clásico fue rápido (0.11 s) pero incapaz de inferir los patrones subyacentes no lineales, arrojando un MAE = 0.17.
* La arquitectura `SAITS` (Transformer), gracias a su mecanismo de auto-atención diagonal, logró mapear las dependencias no lineales tanto temporales como espaciales a un nivel profundo, logrando un MAE = 0.06 (una precisión casi 3 veces superior). Este resultado justifica la penalización en el coste computacional (941 s), demostrando que, ante correlaciones absolutas medias débiles, las arquitecturas profundas son la mejor garantía de calidad del dato de la que disponemos.